# ML Baseline — Gaming Compliance & Risk Intelligence Platform

A **logistic-regression baseline** that predicts the synthetic `IS_LAUNDERING` label from
transaction features **served from Snowflake** — the ML companion to the rule engine, showing a
hybrid rules-plus-ML approach (mirrors the analyst version's baseline). **Synthetic data only.**

> **Optimistic by construction:** the synthetic laundering patterns are generated to match the
> rules, so model scores are illustrative, not production performance.

Run: `pip install -r ../requirements.txt` (includes scikit-learn), then Run All and commit with outputs.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector

# No credentials in this file. Uses ~/.snowflake/connections.toml ('gaming_compliance') or SNOWFLAKE_* env vars.
try:
    conn = snowflake.connector.connect(connection_name=os.environ.get("SNOWFLAKE_CONNECTION", "gaming_compliance"))
except Exception:
    kwargs = dict(account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"],
                  role=os.environ.get("SNOWFLAKE_ROLE", "DATA_ENGINEER"),
                  warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE", "WH_REPORTING"), database="GAMING_COMPLIANCE_DB")
    if os.environ.get("SNOWFLAKE_PASSWORD"): kwargs["password"] = os.environ["SNOWFLAKE_PASSWORD"]
    else: kwargs["authenticator"] = os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser")
    conn = snowflake.connector.connect(**kwargs)

for stmt in ("USE ROLE DATA_ENGINEER", "USE WAREHOUSE WH_REPORTING", "USE DATABASE GAMING_COMPLIANCE_DB"):
    conn.cursor().execute(stmt)

def q(sql):
    cur = conn.cursor()
    try: return cur.execute(sql).fetch_pandas_all()
    finally: cur.close()

q("SELECT CURRENT_ROLE() AS ROLE, CURRENT_WAREHOUSE() AS WH, CURRENT_DATABASE() AS DB")

## 1. Load features + label from Snowflake

In [ ]:
df = q('''SELECT AMOUNT::FLOAT AS AMOUNT,
                 IFF(IS_HIGH_RISK_METHOD,1,0) AS IS_HIGH_RISK_METHOD,
                 IFF(SANCTIONS_FLAG,1,0)      AS SANCTIONS_FLAG,
                 TRANSACTION_TYPE, PAYMENT_FORMAT,
                 IFF(IS_LAUNDERING,1,0)       AS IS_LAUNDERING
          FROM STAGING.STG_TRANSACTIONS WHERE IS_VALID''')
print(df.shape, '| laundering rate:', round(df['IS_LAUNDERING'].mean()*100,1), '%')
df.head()

## 2. Prepare features (one-hot encode categoricals)

In [ ]:
from sklearn.model_selection import train_test_split
y = df['IS_LAUNDERING']
X = pd.get_dummies(df.drop(columns=['IS_LAUNDERING']), columns=['TRANSACTION_TYPE','PAYMENT_FORMAT']).astype(float)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print('features:', list(X.columns))
print('train/test:', Xtr.shape, Xte.shape)

## 3. Train logistic regression + evaluate

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
scaler = StandardScaler().fit(Xtr)
model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(scaler.transform(Xtr), ytr)
proba = model.predict_proba(scaler.transform(Xte))[:,1]
print('ROC-AUC:', round(roc_auc_score(yte, proba), 3))
print(classification_report(yte, (proba>=0.5).astype(int), digits=3))

## 4. ROC curve

In [ ]:
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay.from_predictions(yte, proba, color='#2563eb')
plt.plot([0,1],[0,1],'--',color='#94a3b8'); plt.title('ROC — logistic-regression baseline')
plt.tight_layout(); plt.show()

## 5. Feature importance (coefficients)

In [ ]:
coef = pd.Series(model.coef_[0], index=X.columns).sort_values()
ax = coef.plot.barh(figsize=(7,6), color=['#dc2626' if v<0 else '#16a34a' for v in coef])
ax.set_title('Logistic-regression coefficients (standardized)'); plt.tight_layout(); plt.show()
coef.to_frame('coefficient')

## Findings
_(Fill in after running — ROC-AUC, which features drive the prediction, and how the ML baseline
complements the transparent rule engine in a hybrid approach. Optimistic by construction.)_